In [1]:
import sys, os
from pathlib import Path

# src_path parent must be in sys.path so 'from src.io import ...' resolves
src_path = Path("../../src").resolve()
sys.path.insert(0, str(src_path.parent))

import yaml
cfg_path = src_path.parent / "configs/global.yml"
cfg = yaml.safe_load(open(cfg_path))

SEED      = cfg["seed"]
PROCESSED = (src_path.parent / cfg["paths"]["processed"]).resolve()
RAW       = (src_path.parent / cfg["paths"]["raw"]).resolve()

RAW_FRAMES_DIR = RAW / "hotspot_frames"
ANN_FRAMES_DIR = PROCESSED / "hotspot_frames"

RAW_FRAMES_DIR.mkdir(parents=True, exist_ok=True)
ANN_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

os.environ['MAPILLARY_ACCESS_TOKEN'] = 'MLY|27306421035677736|d506902d0cd89d1953174e7427290ac0'

print(f"PROCESSED : {PROCESSED}")
print(f"RAW       : {RAW}")
print(f"Seed      : {SEED}")

PROCESSED : /home/saumy/Documents/Poor-Visibility-on-Parking-Induced-Congestion/ML/data/processed
RAW       : /home/saumy/Documents/Poor-Visibility-on-Parking-Induced-Congestion/ML/data/raw
Seed      : 42


In [2]:
import importlib, subprocess, sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])

print("ultralytics ready")


ultralytics ready


In [4]:
import json, requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

MAPILLARY_TOKEN = "MLY|27306421035677736|d506902d0cd89d1953174e7427290ac0"
print("All imports OK | Token loaded")

All imports OK | Token loaded


In [6]:
# YOLOv8s — public model, auto-downloads (~22MB)
# Classes relevant to us: car, motorcycle, bus, truck, bicycle
model = YOLO("yolov8s.pt")
print(f"Model loaded: yolov8s")
print(f"Classes ({len(model.names)}): {list(model.names.values())}")


Model loaded: yolov8s
Classes (80): ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [7]:
with open(PROCESSED / "hotspots_summary.json") as f:
    hotspots = json.load(f)

top10 = sorted(hotspots, key=lambda x: x["congestion_impact_score"], reverse=True)[:10]

print("Top 10 clusters targeted:")
for h in top10:
    print(f"  [{h['cluster_id']}] {h['cluster_name']}  "
          f"score={h['congestion_impact_score']}  "
          f"lat={h['center_latitude']:.5f}  lon={h['center_longitude']:.5f}")


Top 10 clusters targeted:
  [2] BTP082 - KR Market Junction Cluster Hub  score=100.0  lat=12.97216  lon=77.57733
  [3] BTP051 - Safina Plaza Junction Cluster Hub  score=88.5  lat=12.98198  lon=77.60800
  [10] New Horizon College Road Area Cluster Hub  score=80.9  lat=12.93344  lon=77.69014
  [5] BTP027 - Modi Bridge Junction Cluster Hub  score=80.8  lat=12.99553  lon=77.55041
  [17] Kadubeesanahalli Underpass Area Cluster Hub  score=80.7  lat=12.94332  lon=77.69772
  [33] 80 Feet Ring Road Area Cluster Hub  score=77.0  lat=13.01030  lon=77.55330
  [34] Sri Venkataranga Ayangar Road Area Cluster Hub  score=76.6  lat=13.00003  lon=77.57120
  [25] Unnamed Road Area Cluster Hub  score=75.4  lat=13.18525  lon=77.68003
  [71] MBT Road Area Cluster Hub  score=72.2  lat=12.99627  lon=77.66859
  [7] MBT Road Area Cluster Hub  score=71.0  lat=13.00850  lon=77.69546


In [8]:
MAPILLARY_API = "https://graph.mapillary.com/images"

def fetch_mapillary_frame(lat, lon, cluster_id, token, delta=0.001):
    for d in [delta, delta * 3]:
        bbox = f"{lon-d},{lat-d},{lon+d},{lat+d}"
        params = {
            "access_token": token,
            "fields": "id,thumb_2048_url,captured_at",
            "bbox": bbox,
            "is_pano": "false",
            "limit": 5,
        }
        resp = requests.get(MAPILLARY_API, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        if data:
            data.sort(key=lambda x: x.get("captured_at", 0), reverse=True)
            img_url = data[0]["thumb_2048_url"]
            img_resp = requests.get(img_url, timeout=30)
            img_resp.raise_for_status()
            img = Image.open(BytesIO(img_resp.content)).convert("RGB")
            save_path = RAW_FRAMES_DIR / f"{cluster_id}_raw.jpg"
            img.save(save_path)
            return str(save_path), data[0]["id"], d
    return None, None, None


frame_meta = {}
for h in top10:
    cid = h["cluster_id"]
    print(f"Fetching [{cid}] {h['cluster_name']} ...", end=" ", flush=True)
    path, img_id, bbox_d = fetch_mapillary_frame(
        h["center_latitude"], h["center_longitude"], cid, MAPILLARY_TOKEN
    )
    if path:
        frame_meta[cid] = {"raw_path": path, "mapillary_image_id": img_id, "bbox_delta": bbox_d}
        print(f"OK  (bbox ±{bbox_d}°)")
    else:
        frame_meta[cid] = {"raw_path": None, "mapillary_image_id": None}
        print("NO IMAGE — skipped")

found = sum(1 for v in frame_meta.values() if v["raw_path"])
print(f"\nFrames fetched: {found}/{len(top10)}")


Fetching [2] BTP082 - KR Market Junction Cluster Hub ... OK  (bbox ±0.001°)
Fetching [3] BTP051 - Safina Plaza Junction Cluster Hub ... OK  (bbox ±0.001°)
Fetching [10] New Horizon College Road Area Cluster Hub ... OK  (bbox ±0.001°)
Fetching [5] BTP027 - Modi Bridge Junction Cluster Hub ... OK  (bbox ±0.003°)
Fetching [17] Kadubeesanahalli Underpass Area Cluster Hub ... OK  (bbox ±0.001°)
Fetching [33] 80 Feet Ring Road Area Cluster Hub ... OK  (bbox ±0.003°)
Fetching [34] Sri Venkataranga Ayangar Road Area Cluster Hub ... OK  (bbox ±0.001°)
Fetching [25] Unnamed Road Area Cluster Hub ... OK  (bbox ±0.001°)
Fetching [71] MBT Road Area Cluster Hub ... OK  (bbox ±0.001°)
Fetching [7] MBT Road Area Cluster Hub ... OK  (bbox ±0.001°)

Frames fetched: 10/10


In [9]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CONF = 0.40
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 0 = first GPU (RTX 4050)
detection_results = {}

for cid, meta in frame_meta.items():
    if not meta["raw_path"]:
        detection_results[cid] = None
        continue

    r = model.predict(
        source=meta["raw_path"],
        conf=CONF,
        device=DEVICE,
        save=False,
        verbose=False
    )[0]

    ann_path = ANN_FRAMES_DIR / f"{cid}_annotated.jpg"
    Image.fromarray(r.plot()[..., ::-1]).save(ann_path)

    class_counts = {}
    for cls_idx in r.boxes.cls.int().tolist():
        label = model.names[cls_idx]
        class_counts[label] = class_counts.get(label, 0) + 1

    confs  = r.boxes.conf.tolist()
    total  = len(confs)
    avg_cf = round(sum(confs) / total, 3) if total else 0.0

    detection_results[cid] = {
        "annotated_image_path": f"hotspot_frames/{cid}_annotated.jpg",
        "frame_source": "Mapillary",
        "mapillary_image_id": meta["mapillary_image_id"],
        "total_vehicles_detected": total,
        "vehicle_breakdown": class_counts,
        "detection_confidence_avg": avg_cf,
    }
    print(f"[{cid}] {total} vehicles | {class_counts} | conf {avg_cf}")

print("\nInference complete.")


CUDA available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU
[2] 9 vehicles | {'car': 2, 'bus': 1, 'person': 5, 'truck': 1} | conf 0.683
[3] 9 vehicles | {'motorcycle': 2, 'person': 3, 'car': 1, 'bus': 2, 'truck': 1} | conf 0.688
[10] 8 vehicles | {'car': 4, 'truck': 1, 'person': 2, 'motorcycle': 1} | conf 0.744
[5] 4 vehicles | {'bus': 2, 'motorcycle': 1, 'person': 1} | conf 0.645
[17] 5 vehicles | {'car': 3, 'person': 1, 'motorcycle': 1} | conf 0.764
[33] 0 vehicles | {} | conf 0.0
[34] 6 vehicles | {'car': 1, 'person': 4, 'umbrella': 1} | conf 0.647
[25] 11 vehicles | {'car': 9, 'truck': 2} | conf 0.74
[71] 6 vehicles | {'car': 4, 'truck': 1, 'bus': 1} | conf 0.546
[7] 3 vehicles | {'bus': 1, 'person': 2} | conf 0.615

Inference complete.


In [10]:
visual_summary = {str(cid): det for cid, det in detection_results.items() if det}

out_path = PROCESSED / "hotspot_visual_summary.json"
with open(out_path, "w") as f:
    json.dump(visual_summary, f, indent=2)

print(f"Saved: {out_path}")
print(f"Clusters with visual evidence: {len(visual_summary)}")
for cid, det in visual_summary.items():
    print(f"  [{cid}] {det['total_vehicles_detected']} vehicles | {det['vehicle_breakdown']}")


Saved: /home/saumy/Documents/Poor-Visibility-on-Parking-Induced-Congestion/ML/data/processed/hotspot_visual_summary.json
Clusters with visual evidence: 10
  [2] 9 vehicles | {'car': 2, 'bus': 1, 'person': 5, 'truck': 1}
  [3] 9 vehicles | {'motorcycle': 2, 'person': 3, 'car': 1, 'bus': 2, 'truck': 1}
  [10] 8 vehicles | {'car': 4, 'truck': 1, 'person': 2, 'motorcycle': 1}
  [5] 4 vehicles | {'bus': 2, 'motorcycle': 1, 'person': 1}
  [17] 5 vehicles | {'car': 3, 'person': 1, 'motorcycle': 1}
  [33] 0 vehicles | {}
  [34] 6 vehicles | {'car': 1, 'person': 4, 'umbrella': 1}
  [25] 11 vehicles | {'car': 9, 'truck': 2}
  [71] 6 vehicles | {'car': 4, 'truck': 1, 'bus': 1}
  [7] 3 vehicles | {'bus': 1, 'person': 2}
